In [0]:
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

In [0]:
cgse = cgse.toPandas()
desempleados = desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
# Leer las tablas de la GEIH 2026 desde Databricks

cgse = spark.read.format("delta").table("workspace.electiva.cgse").toPandas()

ft = spark.read.format("delta").table("workspace.electiva.ft").toPandas()

desempleados = spark.read.format("delta").table("workspace.electiva.desempleados").toPandas()

ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()


In [0]:
# Revisar las columnas disponibles
for nombre, df in {
    "CGSE": cgse,
    "FT": ft,
    "DESOCUPADOS": desempleados,
    "OCUPADOS": ocupados
}.items():
    print(f"\n--- {nombre} ---")
    print(df.columns.tolist())


In [0]:
# Mostrar ejemplos de los datos
display(cgse.head())
display(ft.head())
display(desempleados.head())
display(ocupados.head())

In [0]:
# Buscar columnas relacionadas con las variables del ejercicio

for nombre, df in {
    "CGSE": cgse,
    "FT": ft,
    "DESOCUPADOS": desempleados,
    "OCUPADOS": ocupados
}.items():
    print(f"\n--- {nombre} ---")
    
    for col in df.columns:
        if any(p in col.lower() for p in [
            "edad", "fex", "factor", "expan",
            "mes", "periodo", "año", "ano",
            "ocup", "desocup", "inactiv",
            "pet", "pea", "directorio", "orden"
        ]):
            print(col)

Ejercicio 1

In [0]:
# Revisar las variables disponibles en la tabla FT
print("Valores de MES:")
display(ft["MES"].value_counts(dropna=False))

print("\nValores de PERIODO:")
display(ft["PERIODO"].value_counts(dropna=False))

print("\nValores de PET:")
display(ft["PET"].value_counts(dropna=False))

print("\nValores de FT:")
display(ft["FT"].value_counts(dropna=False))

print("\nValores de FFT:")
display(ft["FFT"].value_counts(dropna=False))

In [0]:
# Filtrar los registros correspondientes a junio de 2026

import pandas as pd
import numpy as np

def filtrar_junio_2026(df):
    periodo = df["PERIODO"].astype(str).str.strip()
    mes = df["MES"].astype(str).str.strip().str.lower()

    filtro_anio = periodo.str.contains("2026", na=False)
    filtro_mes = mes.isin(["6", "06", "junio"])

    return df.loc[filtro_anio & filtro_mes].copy()

cgse_junio = filtrar_junio_2026(cgse)
ocupados_junio = filtrar_junio_2026(ocupados)
desempleados_junio = filtrar_junio_2026(desempleados)

print("Registros CGSE:", len(cgse_junio))
print("Registros ocupados:", len(ocupados_junio))
print("Registros desocupados:", len(desempleados_junio))

In [0]:

# Punto 1: Indicadores del mercado laboral - junio de 2026

# Filtrar junio de 2026
ft_junio = ft[
    (ft["MES"] == 6) &
    (ft["PERIODO"].astype(str).str.startswith("202606"))
].copy()

# Asegurar que el factor de expansión sea numérico
ft_junio["FEX_C18"] = pd.to_numeric(
    ft_junio["FEX_C18"], errors="coerce"
)

# PET: personas en edad de trabajar
pet = (
    ft_junio.loc[ft_junio["PET"] == 1, "FEX_C18"]
    .sum()
)

# PEA: fuerza de trabajo
pea = (
    ft_junio.loc[ft_junio["FT"] == 1, "FEX_C18"]
    .sum()
)

# Población económicamente inactiva
pei = (
    ft_junio.loc[ft_junio["FFT"] == 1, "FEX_C18"]
    .sum()
)

# Población fuera de la edad de trabajar
# Se calcula desde CGSE, usando edad menor de 15 años.
cgse_junio = cgse[
    (cgse["MES"] == 6) &
    (cgse["PERIODO"].astype(str).str.startswith("202606"))
].copy()

cgse_junio["FEX_C18"] = pd.to_numeric(
    cgse_junio["FEX_C18"], errors="coerce"
)
cgse_junio["P6040"] = pd.to_numeric(
    cgse_junio["P6040"], errors="coerce"
)

poblacion_menor_15 = cgse_junio.loc[
    cgse_junio["P6040"] < 15, "FEX_C18"
].sum()

# Ocupados y desocupados
ocupados_junio = ocupados[
    (ocupados["MES"] == 6) &
    (ocupados["PERIODO"].astype(str).str.startswith("202606"))
].copy()

desocupados_junio = desempleados[
    (desempleados["MES"] == 6) &
    (desempleados["PERIODO"].astype(str).str.startswith("202606"))
].copy()

ocupados_junio["FEX_C18"] = pd.to_numeric(
    ocupados_junio["FEX_C18"], errors="coerce"
)
desocupados_junio["FEX_C18"] = pd.to_numeric(
    desocupados_junio["FEX_C18"], errors="coerce"
)

ocup = ocupados_junio["FEX_C18"].sum()
desocup = desocupados_junio["FEX_C18"].sum()

# Tasas
tasa_ocupacion = (ocup / pet) * 100 if pet else 0
tasa_desempleo = (desocup / pea) * 100 if pea else 0

# Resultados
print(f"PET: {pet:,.0f} personas")
print(f"Población fuera de la edad de trabajar: {poblacion_menor_15:,.0f} personas")
print(f"PEA: {pea:,.0f} personas")
print(f"PEI: {pei:,.0f} personas")
print(f"Tasa de ocupación: {tasa_ocupacion:.2f}%")
print(f"Tasa de desempleo: {tasa_desempleo:.2f}%")


In [0]:
#Interpretación de los resultados

print(f"En junio de 2026, la población en edad de trabajar fue de {pet:,.0f} personas.")
print(f"La población fuera de la edad de trabajar fue de {pnoet:,.0f} personas.")
print(f"La población económicamente activa fue de {pea:,.0f} personas.")
print(f"La población económicamente inactiva fue de {pei:,.0f} personas.")
print(f"La tasa de ocupación fue de {tasa_ocupacion:.2f}%.")
print(f"La tasa de desempleo fue de {tasa_desempleo:.2f}%.")

In [0]:
print("Total ponderado de ocupados:", f"{ocup:,.0f}")
print("Total ponderado de desocupados:", f"{desocup:,.0f}")
print("Total ponderado de la PEA:", f"{pea:,.0f}")

print("Ocupados + desocupados:", f"{ocup + desocup:,.0f}")
print("Diferencia frente a la PEA:", f"{(ocup + desocup) - pea:,.0f}")